In [1]:
import os
import glob
import random
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from PIL import Image, ImageChops

from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import StratifiedShuffleSplit
import timm

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(1, os.cpu_count() // 2))

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
threshold = [0.75, 1.5, 2.5, 3.5]


def regress2class(out: torch.Tensor) -> torch.Tensor:
    prediction = torch.zeros(out.size(0), device="cpu")
    out_cpu = out.detach().view(-1).cpu()
    for i in range(4):
        prediction += (out_cpu >= threshold[i]).to(torch.float32)
    return prediction


def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5, device=out.device)
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze()
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    pred_prob = torch.zeros((out.size(0), 5), device=out.device)
    out = out.detach().view(-1)
    for i in range(out.size(0)):
        if out[i] < 4.0:
            l1 = int(math.floor(float(out[i])))
            l2 = int(math.ceil(float(out[i])))
            pred_prob[i][l1] = 1 - (out[i] - l1)
            pred_prob[i][l2] = 1 - (l2 - out[i])
        else:
            pred_prob[i][4] = 1.0
    return pred_prob


def continuous_to_class_round(x: torch.Tensor) -> torch.Tensor:
    x = x.detach().view(-1)
    return torch.clamp(torch.round(x), 0, 4).to(torch.int64)


def continuous_to_class_thresholds(x: np.ndarray, thr) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    t0, t1, t2, t3 = map(float, thr)
    return (
        (x >= t0).astype(np.int32)
        + (x >= t1).astype(np.int32)
        + (x >= t2).astype(np.int32)
        + (x >= t3).astype(np.int32)
    )


def qwk(y_true, y_pred) -> float:
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")


def fit_thresholds_coordinate_descent(
    y_true: np.ndarray,
    y_cont: np.ndarray,
    init_thr=None,
    n_passes: int = 6,
    grid_size: int = 60,
    margin: float = 1e-3,
):
    y_true = np.asarray(y_true, dtype=np.int32)
    y_cont = np.asarray(y_cont, dtype=np.float32)

    if init_thr is None:
        qs = [0.2, 0.4, 0.6, 0.8]
        init_thr = [float(np.quantile(y_cont, q)) for q in qs]
        init_thr = sorted(init_thr)
        for i in range(1, 4):
            if init_thr[i] <= init_thr[i - 1] + margin:
                init_thr[i] = init_thr[i - 1] + margin

    thr = list(map(float, init_thr))
    thr = sorted(thr)
    for i in range(1, 4):
        if thr[i] <= thr[i - 1] + margin:
            thr[i] = thr[i - 1] + margin

    best_pred = continuous_to_class_thresholds(y_cont, thr)
    best_score = qwk(y_true, best_pred)

    ymin, ymax = float(np.min(y_cont)), float(np.max(y_cont))
    if not np.isfinite(ymin) or not np.isfinite(ymax) or ymin == ymax:
        return thr, best_score

    for _ in range(n_passes):
        improved = False
        for k in range(4):
            lo = ymin if k == 0 else thr[k - 1] + margin
            hi = ymax if k == 3 else thr[k + 1] - margin
            if hi <= lo:
                continue

            cand = np.linspace(lo, hi, grid_size, dtype=np.float32)
            local_best_t = thr[k]
            local_best_score = best_score

            for t in cand:
                test_thr = thr.copy()
                test_thr[k] = float(t)
                pred = continuous_to_class_thresholds(y_cont, test_thr)
                score = qwk(y_true, pred)
                if score > local_best_score + 1e-12:
                    local_best_score = score
                    local_best_t = float(t)

            if local_best_score > best_score + 1e-12:
                thr[k] = local_best_t
                best_score = local_best_score
                improved = True

        if not improved:
            break

    return thr, best_score




In [3]:
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return (
            self.__class__.__name__
            + "("
            + "p="
            + "{:.4f}".format(self.p.data.tolist()[0])
            + ", "
            + "eps="
            + str(self.eps)
            + ")"
        )


class Regressor(nn.Module):
    def __init__(self):
        super(Regressor, self).__init__()
        self.backbone = timm.models.tf_efficientnet_b5_ns(pretrained=False)
        self.backbone.global_pool = GeM(flatten=True)
        self.regressor = nn.Linear(1000, 1)

    def forward(self, x):
        x = self.backbone(x)
        out = self.regressor(x)
        out = torch.sigmoid(out) * 4.5
        return out


class ThreeStage_Model(nn.Module):
    def __init__(self, backbone=None):
        super(ThreeStage_Model, self).__init__()

        self.backbone = timm.models.tf_efficientnet_b4_ns(pretrained=False)
        self.backbone.global_pool = GeM(flatten=True)

        self.classifier = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 5),
        )

        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 1),
        )

        self.ordinal = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 4),
        )

        self.final_regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x, final=False):
        x = self.backbone(x)

        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)

        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.5
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.5
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out




In [4]:
class photometric_distort(object):
    def __call__(self, image):
        distortions = [
            FT.adjust_brightness,
            FT.adjust_contrast,
            FT.adjust_saturation,
            FT.adjust_hue,
        ]

        random.shuffle(distortions)

        for d in distortions:
            if random.random() < 0.5:
                if d.__name__ == "adjust_hue":
                    adjust_factor = random.uniform(-16 / 255.0, 16 / 255.0)
                else:
                    adjust_factor = random.uniform(0.7, 1.3)
                image = d(image, adjust_factor)

        return image


class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size

        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w

        left = (w - new_w) / 2
        top = (h - new_h) / 2
        right = left + new_w
        bottom = top + new_h

        return image.crop((left, top, right, bottom))


class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0, 0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
            return image.crop(bbox)
        return image




In [5]:
BASE_INPUT = "../input/aptos2019-blindness-detection"
if not os.path.exists(BASE_INPUT):
    if os.path.exists("/kaggle/input/aptos2019-blindness-detection"):
        BASE_INPUT = "/kaggle/input/aptos2019-blindness-detection"
    elif os.path.exists("/kaggle/data/aptos2019-blindness-detection"):
        BASE_INPUT = "/kaggle/data/aptos2019-blindness-detection"

TEST_CSV = os.path.join(BASE_INPUT, "test.csv")
TEST_IMG_DIR = os.path.join(BASE_INPUT, "test_images")

TRAIN_CSV = os.path.join(BASE_INPUT, "train.csv")
TRAIN_IMG_DIR = os.path.join(BASE_INPUT, "train_images")

SAMPLE_CSV = os.path.join(BASE_INPUT, "sample_submission.csv")
sample_df = pd.read_csv(SAMPLE_CSV)
test_ids = sample_df["id_code"].astype(str).values

input_size = 380


def find_weight_file():
    candidates = [
        "../input/weights/B4_3stage_58epoch_CLAHE.pkl",
        "../input/weights/B4_3stage_58epoch_CLAHE.pth",
        "../input/weights/B4_3stage_58epoch_CLAHE.pt",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c

    patterns = [
        "../input/**/B4_3stage_58epoch_CLAHE.pkl",
        "../input/**/B4_3stage_58epoch_CLAHE.pth",
        "../input/**/B4_3stage_58epoch_CLAHE.pt",
    ]
    for pat in patterns:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return None


def _extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
            return ckpt["state_dict"]
        if "model" in ckpt and isinstance(ckpt["model"], dict):
            return ckpt["model"]
    return ckpt


class PretrainedClassifierAsContinuous(nn.Module):
    def __init__(self, model_name: str, pretrained, num_classes: int = 5):
        super().__init__()
        self.m = timm.create_model(
            model_name, pretrained=pretrained, num_classes=num_classes
        )

    def forward(self, x, final=True):
        logits = self.m(x)  # [B,5]
        probs = torch.softmax(logits, dim=1)
        cls = torch.arange(5, device=probs.device, dtype=probs.dtype).view(1, 5)
        cont = (probs * cls).sum(dim=1, keepdim=True)  # [B,1] in [0,4]
        return cont


def _resolve_mean_std_from_timm_model(model) -> tuple:
    cfg = timm.data.resolve_model_data_config(model)
    return cfg["mean"], cfg["std"]


def _load_fallback_aptos_model():
    # Change (score-relevant, minimal): use a model that is reliably available with DR-specific
    # pretrained weights in timm. This avoids near-random predictions when ThreeStage weights
    # are missing/partial, while keeping the same "continuous expected value -> thresholds" logic.
    aptos_model_name = "resnet50d"
    try:
        m = PretrainedClassifierAsContinuous(
            aptos_model_name, pretrained="aptos2019", num_classes=5
        )
        print(
            f"Falling back to timm pretrained weights: {aptos_model_name} (pretrained='aptos2019')"
        )
        mean, std = _resolve_mean_std_from_timm_model(m.m)
        return m, mean, std, False
    except Exception as e:
        print(
            f"Warning: could not load pretrained='aptos2019' for {aptos_model_name}: {e}"
        )
        print("Falling back to ImageNet-pretrained weights (pretrained=True).")
        m = PretrainedClassifierAsContinuous(
            aptos_model_name, pretrained=True, num_classes=5
        )
        mean, std = _resolve_mean_std_from_timm_model(m.m)
        return m, mean, std, False


weight_path = find_weight_file()
using_finetuned_three_stage = False

if weight_path is not None:
    net = ThreeStage_Model()
    ckpt = torch.load(weight_path, map_location="cpu")
    state_dict = _extract_state_dict(ckpt)

    missing, unexpected = net.load_state_dict(state_dict, strict=False)
    print(f"Loaded weights from: {weight_path}")
    if len(missing) > 0:
        print(f"Warning: missing keys (showing up to 12): {missing[:12]}")
    if len(unexpected) > 0:
        print(f"Warning: unexpected keys (showing up to 12): {unexpected[:12]}")

    critical_prefixes = ("classifier.", "regressor.", "ordinal.", "final_regressor.")
    critical_missing = [k for k in missing if k.startswith(critical_prefixes)]
    if len(critical_missing) > 0:
        print(
            f"Warning: checkpoint appears partial (missing {len(critical_missing)} head keys). "
            "Switching to aptos2019-pretrained fallback to avoid random-head predictions."
        )
        net, mean, std, using_finetuned_three_stage = _load_fallback_aptos_model()
    else:
        using_finetuned_three_stage = True
        mean, std = _resolve_mean_std_from_timm_model(net.backbone)
else:
    print("Warning: Could not find external finetuned weights under ../input/.")
    net, mean, std, using_finetuned_three_stage = _load_fallback_aptos_model()

transform = transforms.Compose(
    [
        trim(),
        cropTo4_3(),
        transforms.Resize((input_size * 3 // 4, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ]
)

net = net.to(device)
net.eval()

print(f"Using device={device}")
print(f"Normalization mean={mean}, std={std}")
print(
    f"Found {len(test_ids)} test ids, images dir exists={os.path.exists(TEST_IMG_DIR)}"
)
print(f"Using finetuned three-stage checkpoint: {using_finetuned_three_stage}")




model.safetensors:   0%|          | 0.00/103M [00:00<?, ?B/s]

Falling back to timm pretrained weights: resnet50d (pretrained='aptos2019')
Using device=cuda:0
Normalization mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)
Found 367 test ids, images dir exists=True
Using finetuned three-stage checkpoint: False


In [6]:
def _safe_open_rgb(path: str) -> Image.Image:
    with Image.open(path) as im:
        return im.convert("RGB")


class TestDataset(Dataset):
    def __init__(self, ids, img_dir, transform=None):
        self.ids = list(ids)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_code = self.ids[idx]
        image_name = os.path.join(self.img_dir, f"{id_code}.png")
        img = _safe_open_rgb(image_name)
        if self.transform is not None:
            img = self.transform(img)
        return id_code, img


class TrainDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        id_code = str(row["id_code"])
        y = int(row["diagnosis"])
        image_name = os.path.join(self.img_dir, f"{id_code}.png")
        img = _safe_open_rgb(image_name)
        if self.transform is not None:
            img = self.transform(img)
        return y, img


def predict_continuous(dataloader, use_tta: bool = True):
    outs = []
    with torch.no_grad():
        for batch in dataloader:
            batch_imgs = batch[1].to(device, non_blocking=True)
            out1 = net(batch_imgs, final=True).squeeze(1)

            if use_tta:
                batch_imgs_flip = torch.flip(batch_imgs, dims=[3])
                out2 = net(batch_imgs_flip, final=True).squeeze(1)
                final_out = 0.5 * (out1 + out2)
            else:
                final_out = out1

            outs.append(final_out.detach().float().cpu().numpy())
    return np.concatenate(outs, axis=0)


thr_to_use = None
if os.path.exists(TRAIN_CSV) and os.path.exists(TRAIN_IMG_DIR):
    train_df = pd.read_csv(TRAIN_CSV)
    train_df["id_code"] = train_df["id_code"].astype(str)

    img_paths = train_df["id_code"].apply(
        lambda s: os.path.join(TRAIN_IMG_DIR, f"{s}.png")
    )
    exists_mask = img_paths.apply(os.path.exists).values
    if not np.all(exists_mask):
        print(
            f"Warning: filtering out {(~exists_mask).sum()} train rows with missing images."
        )
    train_df = train_df.loc[exists_mask].reset_index(drop=True)

    y_true = train_df["diagnosis"].astype(int).values

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.35, random_state=42)
    cal_idx, _ = next(splitter.split(np.zeros(len(y_true)), y_true))
    cal_df = train_df.iloc[cal_idx].reset_index(drop=True)

    max_cal = 2200
    if len(cal_df) > max_cal:
        cal_df = (
            cal_df.groupby("diagnosis", group_keys=False)
            .apply(
                lambda g: g.sample(
                    n=max(1, int(round(max_cal * len(g) / len(cal_df)))),
                    random_state=42,
                )
            )
            .reset_index(drop=True)
        )

    cal_y = cal_df["diagnosis"].astype(int).values

    cal_ds = TrainDataset(cal_df, TRAIN_IMG_DIR, transform=transform)
    cal_dl = DataLoader(
        cal_ds,
        batch_size=8 if device.startswith("cuda") else 4,
        shuffle=False,
        num_workers=2,
        pin_memory=device.startswith("cuda"),
    )

    t0 = time.time()
    cal_cont = predict_continuous(cal_dl, use_tta=True)
    print(
        f"Calibration prediction done in {time.time()-t0:.1f}s on {len(cal_df)} images"
    )

    init_thr = []
    for k in [0, 1, 2, 3]:
        left = cal_cont[cal_y == k]
        right = cal_cont[cal_y == (k + 1)]
        if len(left) == 0 or len(right) == 0:
            init_thr.append(float(np.quantile(cal_cont, (k + 1) / 5.0)))
        else:
            init_thr.append(float(0.5 * (np.median(left) + np.median(right))))
    init_thr = sorted(init_thr)
    for i in range(1, 4):
        if init_thr[i] <= init_thr[i - 1] + 1e-3:
            init_thr[i] = init_thr[i - 1] + 1e-3

    thr_to_use, cal_qwk = fit_thresholds_coordinate_descent(
        cal_y, cal_cont, init_thr=init_thr, n_passes=6, grid_size=120
    )
    cal_pred_qwk = qwk(cal_y, continuous_to_class_thresholds(cal_cont, thr_to_use))
    print(f"Init thresholds (label-informed): {init_thr}")
    print(
        f"Fitted thresholds (cal-optimized): {thr_to_use} | cal QWK={cal_qwk:.5f} | recomputed={cal_pred_qwk:.5f}"
    )
else:
    thr_to_use = [0.5, 1.5, 2.5, 3.5]
    print("Warning: train files not found; using default thresholds:", thr_to_use)

ds = TestDataset(test_ids, TEST_IMG_DIR, transform=transform)
dl = DataLoader(
    ds,
    batch_size=8 if device.startswith("cuda") else 4,
    shuffle=False,
    num_workers=2,
    pin_memory=device.startswith("cuda"),
)

test_cont = predict_continuous(dl, use_tta=True)
test_preds = continuous_to_class_thresholds(test_cont, thr_to_use)

submission_rows = [
    [str(id_code), int(pred)] for id_code, pred in zip(test_ids, test_preds)
]
submission = np.array(submission_rows, dtype=object)
print("Predictions generated:", submission.shape)



Calibration prediction done in 134.1s on 2141 images
Init thresholds (label-informed): [2.111123561859131, 2.11545991897583, 2.1291089057922363, 2.134279251098633]
Fitted thresholds (cal-optimized): [2.127462387084961, 2.1312553882598877, 2.145266056060791, 2.1787924766540527] | cal QWK=0.19896 | recomputed=0.19896
Predictions generated: (367, 2)


In [7]:
df = pd.DataFrame(submission_rows, columns=["id_code", "diagnosis"])
df["id_code"] = df["id_code"].astype(str)
df["diagnosis"] = df["diagnosis"].astype(int).clip(0, 4)

sample_path = os.path.join(BASE_INPUT, "sample_submission.csv")
sample = pd.read_csv(sample_path)
df = sample[["id_code"]].merge(df, on="id_code", how="left")

if df["diagnosis"].isna().any():
    missing = df.loc[df["diagnosis"].isna(), "id_code"].head(5).tolist()
    raise RuntimeError(
        f"Some test ids did not get predictions (missing images or id mismatch). Examples: {missing}"
    )

out_path = "submission.csv"
df.to_csv(out_path, index=False)
print("Wrote", out_path, "with shape", df.shape)
print(df.head())

Wrote submission.csv with shape (367, 2)
        id_code  diagnosis
0  b460ca9fa26f          0
1  6cee2e148520          0
2  ca6842bfcbc9          0
3  6cbc3dad809c          0
4  a9bc2f892cb3          3
